In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import sys
sys.path.insert(0, '..')
import config
from src.data_loader import load_data
from src.pair_selector import run_pair_selection, get_selected_pairs
from src.backtester import run_backtest
from src.risk_manager import compute_net_exposure
from src.metrics import max_drawdown

pd.set_option('display.float_format', '{:.4f}'.format)

In [2]:
# Load data and selected pairs
prices = load_data(force_refresh=False)

try:
    selected = pd.read_csv('../data/selected_pairs.csv')
    print(f"Loaded {len(selected)} selected pairs")
except FileNotFoundError:
    print("Running pair selection...")
    results = run_pair_selection(prices, verbose=False)
    selected = get_selected_pairs(results)
    selected.to_csv('../data/selected_pairs.csv', index=False)

📂 Loading cached data from data/nse_prices.parquet

🧹 Cleaning data...
   Raw shape: (1315, 154)
   Clean shape: (1315, 154)
   Date range: 2021-01-01 to 2026-04-29
   Trading days: 1315
   ✅ No issues found

💾 Saved to data/nse_prices.parquet
Loaded 8 selected pairs


## 1. Run the Full Backtest

This runs the complete pipeline:
- Generates signals on the test period (last 1 year)
- Computes daily P&L with 0.05% per-leg transaction costs + 0.03% slippage
- Builds equity curves
- Extracts every individual trade
- Computes Sharpe, CAGR, max drawdown, win rate, etc.

In [3]:
# Run backtest — takes 1-2 minutes
bt = run_backtest(prices, selected, verbose=True)

BACKTEST ENGINE

📡 Step 1: Generating signals...
📅 Signal generation period:
   Warmup: 2024-05-06 to 2025-04-24
   Test:   2025-04-24 to 2026-04-29

   [1/8] M&M-ASHOKLEY (HL=16d, β=1.851)
      Signals: 6 entries | Long 32d | Short 76d | Flat 144d

   [2/8] APOLLOHOSP-MAXHEALTH (HL=21d, β=0.484)
      Signals: 7 entries | Long 82d | Short 85d | Flat 85d

   [3/8] NTPC-NHPC (HL=18d, β=0.942)
      Signals: 6 entries | Long 1d | Short 151d | Flat 100d

   [4/8] BANDHANBNK-FEDERALBNK (HL=16d, β=-0.656)
      Signals: 8 entries | Long 85d | Short 97d | Flat 70d

   [5/8] MARICO-UBL (HL=22d, β=0.790)
      Signals: 2 entries | Long 0d | Short 50d | Flat 202d

   [6/8] MARUTI-TVSMOTOR (HL=22d, β=0.359)
      Signals: 5 entries | Long 86d | Short 40d | Flat 126d

   [7/8] HDFCAMC-ICICIGI (HL=19d, β=1.594)
      Signals: 7 entries | Long 125d | Short 68d | Flat 59d

   [8/8] KOTAKBANK-INDUSINDBK (HL=19d, β=-0.049)
      Signals: 7 entries | Long 79d | Short 67d | Flat 106d

✅ Generated signa

## 2. Equity Curve: Strategy vs Nifty 50

The most important chart. Our strategy equity curve vs a simple
buy-and-hold of the Nifty 50 index.

What to look for:
- Does the strategy grow steadily or in bursts?
- How deep are the drawdowns compared to Nifty?
- Does the strategy behave differently during market crashes?

In [4]:
fig = go.Figure()

# Strategy equity
fig.add_trace(go.Scatter(
    x=bt['portfolio_equity'].index,
    y=bt['portfolio_equity'].values,
    name='Pairs Strategy',
    line=dict(color='#00d4aa', width=2.5)
))

# Benchmark equity
fig.add_trace(go.Scatter(
    x=bt['benchmark_equity'].index,
    y=bt['benchmark_equity'].values,
    name='Nifty 50 (Buy & Hold)',
    line=dict(color='#ff6b6b', width=2, dash='dash')
))

fig.add_hline(y=1.0, line_dash='dot', line_color='gray', opacity=0.3)

fig.update_layout(
    title='Strategy vs Benchmark — Equity Curves',
    xaxis_title='Date', yaxis_title='Growth of ₹1',
    height=500, template='plotly_dark',
    legend=dict(x=0.02, y=0.98),
)
fig.show()

print(f"\n📊 Strategy: {bt['portfolio_metrics']['total_return']:.2f}% total return")
print(f"📊 Nifty 50: {bt['benchmark_metrics']['total_return']:.2f}% total return")


📊 Strategy: 21.55% total return
📊 Nifty 50: -0.62% total return


## 3. Drawdown Analysis

Drawdown shows how much you would have lost from any peak.
Lower (less negative) drawdowns = smoother ride.

In [5]:
# Drawdown chart
strat_dd = max_drawdown(bt['portfolio_returns'].dropna())['drawdown_series']
bench_dd = max_drawdown(bt['benchmark_returns'].dropna())['drawdown_series']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=strat_dd.index, y=strat_dd.values * 100,
    name='Strategy Drawdown', fill='tozeroy',
    line=dict(color='#00d4aa', width=1),
    fillcolor='rgba(0,212,170,0.15)'
))
fig.add_trace(go.Scatter(
    x=bench_dd.index, y=bench_dd.values * 100,
    name='Nifty 50 Drawdown', fill='tozeroy',
    line=dict(color='#ff6b6b', width=1),
    fillcolor='rgba(255,107,107,0.15)'
))

fig.update_layout(
    title='Drawdown Comparison',
    xaxis_title='Date', yaxis_title='Drawdown (%)',
    height=400, template='plotly_dark',
)
fig.show()

print(f"Strategy Max Drawdown: {bt['portfolio_metrics']['max_drawdown']:.2f}%")
print(f"Nifty 50 Max Drawdown: {bt['benchmark_metrics']['max_drawdown']:.2f}%")

Strategy Max Drawdown: -3.37%
Nifty 50 Max Drawdown: -15.18%


## 4. Performance Metrics — Side by Side

In [6]:
# Compare strategy vs benchmark
comparison = pd.DataFrame({
    'Metric': ['Total Return (%)', 'CAGR (%)', 'Sharpe Ratio', 
               'Max Drawdown (%)', 'Volatility (%)', 'Calmar Ratio'],
    'Pairs Strategy': [
        bt['portfolio_metrics']['total_return'],
        bt['portfolio_metrics']['cagr'],
        bt['portfolio_metrics']['sharpe_ratio'],
        bt['portfolio_metrics']['max_drawdown'],
        bt['portfolio_metrics']['volatility'],
        bt['portfolio_metrics']['calmar_ratio'],
    ],
    'Nifty 50 B&H': [
        bt['benchmark_metrics']['total_return'],
        bt['benchmark_metrics']['cagr'],
        bt['benchmark_metrics']['sharpe_ratio'],
        bt['benchmark_metrics']['max_drawdown'],
        bt['benchmark_metrics']['volatility'],
        bt['benchmark_metrics']['calmar_ratio'],
    ],
}).set_index('Metric')

print("\n" + "="*50)
print("STRATEGY vs BENCHMARK")
print("="*50)
comparison


STRATEGY vs BENCHMARK


,Pairs Strategy,Nifty 50 B&H
Metric,,
Total Return (%),21.5500,-0.6200
CAGR (%),21.5500,-0.6200
Sharpe Ratio,1.9100,-0.4300
Max Drawdown (%),-3.3700,-15.1800
Volatility (%),7.2300,13.2600
Calmar Ratio,6.4000,-0.0400


## 5. Per-Pair Breakdown

In [7]:
# Per-pair metrics table
pair_summary = []
for pair_name, metrics in bt['pair_metrics'].items():
    row = {'Pair': pair_name}
    row.update(metrics)
    pair_summary.append(row)

pair_df = pd.DataFrame(pair_summary).set_index('Pair')
display_cols = ['total_return', 'sharpe_ratio', 'max_drawdown', 'n_trades', 'win_rate', 'profit_factor']
available_cols = [c for c in display_cols if c in pair_df.columns]
pair_df[available_cols]

,total_return,sharpe_ratio,max_drawdown,n_trades,win_rate,profit_factor
Pair,,,,,,
M&M-ASHOKLEY,22.5500,0.7700,-10.0200,5,80.0000,3.4800
APOLLOHOSP-MAXHEALTH,1.4200,-0.2700,-10.7600,6,66.7000,1.7200
NTPC-NHPC,-15.6400,-1.3800,-22.0400,5,20.0000,0.3500
BANDHANBNK-FEDERALBNK,70.9800,1.6100,-26.4900,7,71.4000,6.6100
MARICO-UBL,6.2600,0.0500,-4.1100,2,100.0000,inf
MARUTI-TVSMOTOR,20.9200,0.8500,-8.4500,5,60.0000,4.1600
HDFCAMC-ICICIGI,46.5600,1.2200,-17.8800,7,85.7000,195.7100
KOTAKBANK-INDUSINDBK,20.4500,0.7800,-14.6500,6,66.7000,3.3900


In [8]:
# Per-pair equity curves
fig = go.Figure()
for pair_name, equity in bt['pair_equity'].items():
    fig.add_trace(go.Scatter(
        x=equity.index, y=equity.values,
        name=pair_name, mode='lines',
        line=dict(width=1.5)
    ))

fig.add_hline(y=1.0, line_dash='dot', line_color='gray', opacity=0.3)
fig.update_layout(
    title='Per-Pair Equity Curves',
    xaxis_title='Date', yaxis_title='Growth of ₹1',
    height=500, template='plotly_dark',
)
fig.show()

## 6. Trade Analysis

In [9]:
trades = bt['all_trades']

if len(trades) > 0:
    print(f"Total trades: {len(trades)}")
    print(f"Long trades: {(trades['direction'] == 'Long').sum()}")
    print(f"Short trades: {(trades['direction'] == 'Short').sum()}")
    print(f"\nAverage holding period: {trades['holding_days'].mean():.1f} days")
    print(f"Median holding period: {trades['holding_days'].median():.1f} days")
    print(f"\nProfitable trades: {(trades['trade_return'] > 0).sum()} ({(trades['trade_return'] > 0).mean()*100:.1f}%)")
    print(f"Losing trades: {(trades['trade_return'] <= 0).sum()} ({(trades['trade_return'] <= 0).mean()*100:.1f}%)")
    
    print(f"\nRecent trades:")
    display(trades.tail(15))
else:
    print("No trades were generated. Check pair selection and signal thresholds.")

Total trades: 43
Long trades: 18
Short trades: 25

Average holding period: 36.6 days
Median holding period: 41.0 days

Profitable trades: 29 (67.4%)
Losing trades: 14 (32.6%)

Recent trades:


,pair,direction,entry_date,exit_date,entry_z,exit_z,holding_days,trade_return
28,MARUTI-TVSMOTOR,Short,2026-01-14,2026-01-22,2.2100,0.3400,8,-0.0131
29,MARUTI-TVSMOTOR,Short,2026-02-05,2026-03-17,3.4000,0.3800,40,0.2114
30,HDFCAMC-ICICIGI,Long,2025-04-24,2025-05-06,-0.8000,-0.4300,12,0.0042
31,HDFCAMC-ICICIGI,Long,2025-05-29,2025-07-22,-2.3900,-1.6700,54,0.0852
32,HDFCAMC-ICICIGI,Long,2025-07-23,2025-09-02,-1.6700,-0.1700,41,0.1197
33,HDFCAMC-ICICIGI,Short,2025-09-12,2025-11-10,2.9200,1.5600,59,0.0899
34,HDFCAMC-ICICIGI,Short,2025-11-11,2025-12-23,1.5300,0.4700,42,0.0295
35,HDFCAMC-ICICIGI,Long,2026-01-28,2026-03-24,-2.3900,-2.0700,55,-0.0023
36,HDFCAMC-ICICIGI,Long,2026-03-25,2026-04-21,-1.9500,-0.4800,27,0.1187
37,KOTAKBANK-INDUSINDBK,Short,2025-04-24,2025-05-15,1.4100,0.6100,21,0.0713


In [10]:
# Trade P&L distribution
if len(trades) > 0:
    fig = make_subplots(rows=1, cols=2, subplot_titles=[
        'Trade P&L Distribution', 'Holding Period Distribution'
    ])
    
    # P&L histogram
    fig.add_trace(go.Histogram(
        x=trades['trade_return'], nbinsx=30,
        marker_color='#6b9fff', name='P&L'
    ), row=1, col=1)
    fig.add_vline(x=0, row=1, col=1, line_color='white', line_dash='dash')
    
    # Holding period histogram
    fig.add_trace(go.Histogram(
        x=trades['holding_days'], nbinsx=20,
        marker_color='#b388ff', name='Holding Days'
    ), row=1, col=2)
    
    fig.update_layout(height=400, template='plotly_dark', showlegend=False)
    fig.show()

## 7. Net Market Exposure

Are we truly market-neutral? Net exposure should hover near zero.

In [11]:
# Net exposure over time
exposure = compute_net_exposure(
    bt['signals'], 
    bt['position_sizes']['capital_per_pair']
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=exposure.index, y=exposure['net_exposure'],
    name='Net Exposure', fill='tozeroy',
    line=dict(color='#b388ff', width=1),
    fillcolor='rgba(179,136,255,0.2)'
))
fig.add_trace(go.Scatter(
    x=exposure.index, y=exposure['gross_exposure'],
    name='Gross Exposure',
    line=dict(color='#ff9f43', width=1, dash='dot')
))
fig.add_hline(y=0, line_color='gray', opacity=0.3)

fig.update_layout(
    title='Portfolio Exposure Over Time',
    xaxis_title='Date', yaxis_title='Exposure (₹)',
    height=400, template='plotly_dark',
)
fig.show()

avg_net = exposure['net_exposure'].mean()
avg_gross = exposure['gross_exposure'].mean()
print(f"Average net exposure: ₹{avg_net:,.0f} (should be near 0)")
print(f"Average gross exposure: ₹{avg_gross:,.0f}")

Average net exposure: ₹-6,857 (should be near 0)
Average gross exposure: ₹53,524


## 8. Impact of Transaction Costs

How much did transaction costs eat into our returns?

In [12]:
# Gross vs net returns comparison
gross_returns = pd.DataFrame({
    name: ret['gross_return'] for name, ret in bt['pair_returns'].items()
}).mean(axis=1).fillna(0)

gross_equity = (1 + gross_returns).cumprod()
net_equity = bt['portfolio_equity']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=gross_equity.index, y=gross_equity.values,
    name='Gross (no costs)', line=dict(color='#00d4aa', width=2)
))
fig.add_trace(go.Scatter(
    x=net_equity.index, y=net_equity.values,
    name='Net (with costs)', line=dict(color='#ff6b6b', width=2)
))

fig.update_layout(
    title='Impact of Transaction Costs',
    xaxis_title='Date', yaxis_title='Growth of ₹1',
    height=400, template='plotly_dark',
)
fig.show()

total_costs = sum(
    ret['cost'].sum() for ret in bt['pair_returns'].values()
)
print(f"\nTotal transaction costs: {total_costs*100:.2f}% of capital")
print(f"Gross return: {(gross_equity.iloc[-1]-1)*100:.2f}%")
print(f"Net return:   {(net_equity.iloc[-1]-1)*100:.2f}%")
print(f"Cost drag:    {(gross_equity.iloc[-1]-net_equity.iloc[-1])*100:.2f}%")


Total transaction costs: 11.05% of capital
Gross return: 23.24%
Net return:   21.55%
Cost drag:    1.69%


## 9. Save Backtest Results

In [14]:
# Save key results
bt['all_trades'].to_csv('../data/trade_log.csv', index=False)
bt['portfolio_equity'].to_csv('../data/portfolio_equity.csv')

# Save metrics summary
metrics_summary = pd.DataFrame(bt['pair_metrics']).T
metrics_summary.loc['PORTFOLIO'] = bt['portfolio_metrics']
metrics_summary.loc['NIFTY_50'] = bt['benchmark_metrics']
metrics_summary.to_csv('../data/backtest_metrics.csv')

print("✅ Saved")

✅ Saved
